# Reproduce the CisFalcon flagship number in your browser

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/belumume/cisfalcon/blob/main/reproduce_flagship.ipynb)

One click, no install, no GPU, no API key. This fetches the committed per-design cross-lab scores (93,435 AI-designed enhancers from an independent lab, Gosai/Tewhey 2024) straight from GitHub and re-derives the headline numbers with a plain rank-sum AUROC in pure numpy, so you can watch **AUROC 0.8013** print rather than take it on faith. Full method and every caveat: `PREREG.md`.

In [ ]:
# CisFalcon flagship, reproduced from the committed cross-lab scores.
import io, urllib.request, csv
import numpy as np

URL = "https://raw.githubusercontent.com/belumume/cisfalcon/main/data/gosai_designed/designed_scored.csv"
raw = urllib.request.urlopen(urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})).read().decode()
rows = list(csv.DictReader(io.StringIO(raw)))
y = np.array([int(r["measured_fail"]) for r in rows])       # 1 = wet-lab specificity failure
gap = np.array([float(r["pred_gap"]) for r in rows])        # CisFalcon predicted specificity gap
n = len(y); base = y.mean()

def auroc(score, labels):
    """Rank-sum (Mann-Whitney) AUROC; higher score = higher predicted failure risk."""
    order = np.argsort(score, kind="mergesort")
    rank = np.empty(len(score)); rank[order] = np.arange(1, len(score) + 1)
    n1 = labels.sum(); n0 = len(labels) - n1
    return float((rank[labels == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

auc = auroc(-gap, y)
order = np.argsort(gap)                                     # riskiest (smallest gap) first
ys = y[order]
k2 = int(round(0.02 * n))
ppv2 = ys[:k2].mean()
safe_half = ys[n // 2:].mean()
rng = np.random.default_rng(0)
auc_random = auroc(rng.random(n), y)

print(f"designs (independent lab, zero training overlap): {n:,}")
print(f"wet-lab specificity-failure base rate           : {base*100:.2f}%  (~1 in {1/base:.0f})")
print(f"cross-lab AUROC                                 : {auc:.4f}")
print(f"  random-score baseline                         : {auc_random:.4f}")
print(f"riskiest 2% flagged, truly fail                 : {ppv2*100:.1f}%  ({ppv2/base:.2f}x base)")
print(f"synthesize the safest half, failure rate        : {safe_half*100:.2f}%  ({(1-safe_half/base)*100:.0f}% fewer)")
print()
print("Everything above is POOLED across cells and generators, which is how a mixed batch arrives.")
print("The next cell conditions it and reports the sequence-free null.")
print("The fully-conditioned per-sequence AUROC is 0.66; see cell_prior_baseline.py and PREREG.md.")


## The pooled numbers above are not the honest ones

Everything printed above is **pooled** across cells and generators. A pooled figure partly measures
how far apart the strata sit, so it must not be read as per-design skill. The cell below re-derives,
from the same fetched file, the conditioned figure and the null that makes the point: a rule using
**no sequence at all**, scoring each design by its stratum's base rate, beats the pooled reduction.
Read the null first. Full method: `triage_conditioning_check.py`, `PREREG-ERRATA.md`.


In [ ]:
# Conditioning, from the same rows already fetched above. Still pure numpy.
MIN_PER_CLASS = 10                      # the repo's criterion: >=10 fail and >=10 pass per stratum
cellv = np.array([r["target_cell"] for r in rows])
methv = np.array([r["method"] for r in rows])

def safest_half_rate(g_gap, g_y):
    """Rank safest-first (largest predicted gap = safest), take the safer half."""
    o = np.argsort(g_gap, kind="mergesort")
    return g_y[o][len(g_y) // 2:].mean()

strata = sorted(set(zip(cellv.tolist(), methv.tolist())))
reds, kept = [], 0
for c, m in strata:
    sel = (cellv == c) & (methv == m)
    gy, gg = y[sel], gap[sel]
    if gy.sum() < MIN_PER_CLASS or (len(gy) - gy.sum()) < MIN_PER_CLASS:
        continue
    reds.append(100.0 * (1 - safest_half_rate(gg, gy) / gy.mean()))
    kept += len(gy)
macro = sum(reds) / len(reds)
pooled = 100.0 * (1 - safest_half_rate(gap, y) / base)

# Sequence-free null: score every design by its stratum base rate, nothing else.
prior = np.zeros(n)
for c, m in strata:
    sel = (cellv == c) & (methv == m)
    prior[sel] = y[sel].mean()
# Every design inside a stratum ties, so the safer half is decided purely by tie-break. That makes
# this figure mildly seed-dependent: across 12 seeds it runs 90.1% to 90.8%. Reported as measured
# rather than as a point estimate, because a number that moves with the RNG should say so.
nulls = []
for s in range(12):
    sh = np.random.default_rng(s).permutation(n)
    nulls.append(100.0 * (1 - safest_half_rate(-prior[sh], y[sh]) / base))

print(f"strata with >=10 of each class                  : {len(reds)} "
      f"({kept:,} of {n:,} designs, {100*kept/n:.0f}%)")
print(f"safest-half failure reduction, POOLED           : {pooled:.0f}%   <- not per-design skill")
print(f"safest-half failure reduction, macro-conditioned: {macro:.0f}%   <- the honest one")
print(f"sequence-free stratum-prior null, pooled        : {sum(nulls)/len(nulls):.1f}% "
      f"(range {min(nulls):.1f}-{max(nulls):.1f} over 12 tie-break seeds)")
print()
print("The null uses no sequence information and still beats the pooled reduction, so the pooled")
print("number reflects how far apart the strata sit. The conditioned figure is the per-design one.")
